# Notebook 3: Campaign Performance & ROI Analysis

## Overview
This notebook analyzes campaign performance metrics and ROI across different marketing channels and customer segments. We examine campaign acceptance rates, channel effectiveness, and provide recommendations for budget reallocation to maximize ROI.

**Author:** Stephen Drani  
**Date:** March 2026

## 3.1 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Ensure visualization directory exists
os.makedirs('../visualizations/', exist_ok=True)

## 3.2 Load Segmented Data

In [ ]:
# Load the segmented marketing data from Notebook 02
df = pd.read_csv('../data/processed/marketing_segmented.csv')

print(f"Data Shape: {df.shape}")
print(f"\\nFirst few rows:")
print(df.head())
print(f"\\nColumn Names:")
print(df.columns.tolist())
print(f"\\nData Types:")
print(df.dtypes)

## 3.3 Overall Campaign Acceptance Rates

In [ ]:
# Define campaign columns
campaign_cols = ['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'Response']

# Calculate acceptance rates for each campaign
acceptance_rates = {}

for col in campaign_cols:
    if col in df.columns:
        acceptance_rates[col] = (df[col].sum() / len(df)) * 100

# Create a dataframe for better visualization
acceptance_df = pd.DataFrame(list(acceptance_rates.items()), 
                              columns=['Campaign', 'Acceptance Rate (%)'])
acceptance_df = acceptance_df.sort_values('Acceptance Rate (%)', ascending=False)

print("Campaign Acceptance Rates:")
print(acceptance_df)
print(f"\\nBest Campaign: {acceptance_df.iloc[0]['Campaign']} ({acceptance_df.iloc[0]['Acceptance Rate (%)']:.2f}%)")
print(f"Worst Campaign: {acceptance_df.iloc[-1]['Campaign']} ({acceptance_df.iloc[-1]['Acceptance Rate (%)']:.2f}%)")

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
colors = ['green' if x == acceptance_df.iloc[0]['Acceptance Rate (%)'] else 
          'red' if x == acceptance_df.iloc[-1]['Acceptance Rate (%)'] else 'steelblue' 
          for x in acceptance_df['Acceptance Rate (%)']]
bars = ax.bar(acceptance_df['Campaign'], acceptance_df['Acceptance Rate (%)'], color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Acceptance Rate (%)', fontsize=12, fontweight='bold')
ax.set_xlabel('Campaign', fontsize=12, fontweight='bold')
ax.set_title('Campaign Acceptance Rates - Overall Performance', fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2f}%',
            ha='center', va='bottom', fontweight='bold')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../visualizations/03_campaign_acceptance_rates.png', dpi=300, bbox_inches='tight')
plt.show()

## 3.4 Campaign Performance by Customer Segment

In [ ]:
# Calculate campaign acceptance rates by segment
if 'RFM_Segment' in df.columns:
    segment_campaign_rates = df.groupby('RFM_Segment')[campaign_cols].mean() * 100
    
    print("Campaign Acceptance Rates by Segment:")
    print(segment_campaign_rates.round(2))
    
    # Heatmap visualization
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.heatmap(segment_campaign_rates, annot=True, fmt='.2f', cmap='YlGn', 
                cbar_kws={'label': 'Acceptance Rate (%)'}, ax=ax, linewidths=0.5)
    ax.set_title('Campaign Acceptance Rates by Customer Segment', fontsize=14, fontweight='bold', pad=20)
    ax.set_ylabel('Customer Segment', fontsize=12, fontweight='bold')
    ax.set_xlabel('Campaign', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../visualizations/03_campaign_by_segment_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Grouped bar chart
    fig, ax = plt.subplots(figsize=(14, 6))
    segment_campaign_rates.plot(kind='bar', ax=ax, width=0.8)
    ax.set_title('Campaign Acceptance Rates by Customer Segment - Grouped View', fontsize=14, fontweight='bold', pad=20)
    ax.set_ylabel('Acceptance Rate (%)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Customer Segment', fontsize=12, fontweight='bold')
    ax.legend(title='Campaign', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../visualizations/03_campaign_by_segment_grouped.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("RFM_Segment column not found in dataset")

## 3.5 Total Campaigns Accepted by Segment

In [ ]:
# Calculate total campaigns accepted by segment
if 'Total_Campaigns_Accepted' in df.columns and 'RFM_Segment' in df.columns:
    segment_total_campaigns = df.groupby('RFM_Segment')['Total_Campaigns_Accepted'].mean()
    
    print("Average Total Campaigns Accepted by Segment:")
    print(segment_total_campaigns.sort_values(ascending=False))
    
    # Bar chart
    fig, ax = plt.subplots(figsize=(12, 6))
    segment_total_campaigns_sorted = segment_total_campaigns.sort_values(ascending=False)
    bars = ax.bar(range(len(segment_total_campaigns_sorted)), segment_total_campaigns_sorted.values, 
                   color='steelblue', alpha=0.7, edgecolor='black')
    ax.set_xticks(range(len(segment_total_campaigns_sorted)))
    ax.set_xticklabels(segment_total_campaigns_sorted.index, rotation=45, ha='right')
    ax.set_ylabel('Average Total Campaigns Accepted', fontsize=12, fontweight='bold')
    ax.set_xlabel('Customer Segment', fontsize=12, fontweight='bold')
    ax.set_title('Responsiveness by Segment: Average Total Campaigns Accepted', fontsize=14, fontweight='bold', pad=20)
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}',
                ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../visualizations/03_total_campaigns_by_segment.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\\nMost responsive segment: {segment_total_campaigns_sorted.index[0]} ({segment_total_campaigns_sorted.values[0]:.2f} campaigns)")
    print(f"Least responsive segment: {segment_total_campaigns_sorted.index[-1]} ({segment_total_campaigns_sorted.values[-1]:.2f} campaigns)")
else:
    print("Required columns not found in dataset")

## 3.6 Channel Analysis

We analyze performance across three main marketing channels:
- **Web**: Direct online purchases
- **Catalog**: Mail/print catalog purchases
- **Store**: In-store purchases

In [ ]:
# Define channel columns
channel_cols = {'Web': 'NumWebPurchases', 'Catalog': 'NumCatalogPurchases', 'Store': 'NumStorePurchases'}

# Channel analysis
channel_analysis = {}
for channel, col in channel_cols.items():
    if col in df.columns:
        purchases = df[col].sum()
        avg_per_customer = df[col].mean()
        channel_analysis[channel] = {'Total Purchases': purchases, 'Avg per Customer': avg_per_customer}

channel_df = pd.DataFrame(channel_analysis).T
print("Channel Purchase Summary:")
print(channel_df.round(2))

# Visualization: Total Purchases by Channel
fig, ax = plt.subplots(figsize=(12, 6))

channels = list(channel_analysis.keys())
purchases = [channel_analysis[c]['Total Purchases'] for c in channels]
colors_purchases = ['#1f77b4', '#ff7f0e', '#2ca02c']
bars = ax.bar(channels, purchases, color=colors_purchases, alpha=0.7, edgecolor='black')
ax.set_ylabel('Total Number of Purchases', fontsize=12, fontweight='bold')
ax.set_title('Total Purchases by Channel', fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3)

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height):,}',
            ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('../visualizations/03_channel_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 3.7 Channel Usage by Segment

In [ ]:
# Channel usage by segment
if 'RFM_Segment' in df.columns:
    channel_by_segment = df.groupby('RFM_Segment')[list(channel_cols.values())].mean()
    channel_by_segment.columns = list(channel_cols.keys())
    
    print("Average Channel Usage by Segment:")
    print(channel_by_segment.round(2))
    
    # Grouped bar chart
    fig, ax = plt.subplots(figsize=(12, 6))
    colors_purchases = ['#1f77b4', '#ff7f0e', '#2ca02c']
    channel_by_segment.plot(kind='bar', ax=ax, width=0.8, color=colors_purchases, alpha=0.8, edgecolor='black')
    ax.set_title('Channel Usage by Customer Segment', fontsize=14, fontweight='bold', pad=20)
    ax.set_ylabel('Average Number of Purchases', fontsize=12, fontweight='bold')
    ax.set_xlabel('Customer Segment', fontsize=12, fontweight='bold')
    ax.legend(title='Channel', fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('../visualizations/03_channel_by_segment.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("RFM_Segment column not found")

## 3.8 ROI Calculation

### Cost Assumptions (Simulated for Analysis)
- **Web Channel**: $5 cost per transaction (digital marketing, email campaigns)
- **Catalog Channel**: $15 cost per transaction (printing, postage, logistics)
- **Store Channel**: $3 cost per transaction (in-store marketing, displays)

ROI Metrics:
- **ROI (%)** = ((Revenue - Cost) / Cost) × 100
- **Revenue per Channel**: MntTotal distributed proportionally based on purchase volumes
- **Net Profit** = Total Revenue - Total Cost

In [ ]:
# Define cost assumptions (in dollars per transaction)
cost_per_transaction = {
    'Web': 5,
    'Catalog': 15,
    'Store': 3
}

# Calculate total revenue distributed proportionally across channels based on purchase volumes
total_revenue = df['MntTotal'].sum() if 'MntTotal' in df.columns else 0

# Calculate purchase volumes
total_web_purchases = df['NumWebPurchases'].sum() if 'NumWebPurchases' in df.columns else 0
total_catalog_purchases = df['NumCatalogPurchases'].sum() if 'NumCatalogPurchases' in df.columns else 0
total_store_purchases = df['NumStorePurchases'].sum() if 'NumStorePurchases' in df.columns else 0

total_purchases = total_web_purchases + total_catalog_purchases + total_store_purchases

# Distribute revenue proportionally
revenue_by_channel = {}
cost_by_channel = {}

if total_purchases > 0:
    revenue_by_channel['Web'] = total_revenue * (total_web_purchases / total_purchases)
    revenue_by_channel['Catalog'] = total_revenue * (total_catalog_purchases / total_purchases)
    revenue_by_channel['Store'] = total_revenue * (total_store_purchases / total_purchases)
else:
    revenue_by_channel = {'Web': 0, 'Catalog': 0, 'Store': 0}

# Calculate costs
cost_by_channel['Web'] = total_web_purchases * cost_per_transaction['Web']
cost_by_channel['Catalog'] = total_catalog_purchases * cost_per_transaction['Catalog']
cost_by_channel['Store'] = total_store_purchases * cost_per_transaction['Store']

# Calculate ROI by channel
roi_results = {}

for channel in ['Web', 'Catalog', 'Store']:
    revenue = revenue_by_channel.get(channel, 0)
    cost = cost_by_channel.get(channel, 0)
    net_profit = revenue - cost
    
    roi_pct = ((revenue - cost) / cost * 100) if cost > 0 else 0
    revenue_per_dollar = (revenue / cost) if cost > 0 else 0
    
    purchase_col = channel_cols.get(channel)
    num_purchases = df[purchase_col].sum() if purchase_col in df.columns else 0
    
    roi_results[channel] = {
        'Total Revenue ($)': revenue,
        'Total Cost ($)': cost,
        'Net Profit ($)': net_profit,
        'ROI (%)': roi_pct,
        'Revenue per $1 Spent': revenue_per_dollar,
        'Total Purchases': int(num_purchases)
    }

roi_df = pd.DataFrame(roi_results).T

print("\\n" + "="*80)
print("ROI ANALYSIS BY CHANNEL")
print("="*80)
print(roi_df.round(2))
print("="*80)

# Visualization: ROI comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ROI percentage
roi_channels = roi_df.index.tolist()
roi_values = roi_df['ROI (%)'].values
roi_colors = ['green' if x > 0 else 'red' for x in roi_values]

bars1 = ax1.bar(roi_channels, roi_values, color=roi_colors, alpha=0.7, edgecolor='black')
ax1.set_ylabel('ROI (%)', fontsize=12, fontweight='bold')
ax1.set_title('Return on Investment (ROI) by Channel', fontsize=13, fontweight='bold', pad=15)
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax1.grid(axis='y', alpha=0.3)

for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%',
            ha='center', va='bottom' if height > 0 else 'top', fontweight='bold')

# Revenue per dollar spent
revenue_per_dollar_vals = roi_df['Revenue per $1 Spent'].values
bars2 = ax2.bar(roi_channels, revenue_per_dollar_vals, color=['#2ca02c' if x > 1 else '#d62728' for x in revenue_per_dollar_vals], 
                alpha=0.7, edgecolor='black')
ax2.set_ylabel('Revenue per $1 Spent', fontsize=12, fontweight='bold')
ax2.set_title('Revenue Efficiency by Channel', fontsize=13, fontweight='bold', pad=15)
ax2.axhline(y=1, color='red', linestyle='--', linewidth=1.5, label='Break-even')
ax2.grid(axis='y', alpha=0.3)
ax2.legend()

for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'${height:.2f}',
            ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('../visualizations/03_roi_by_channel.png', dpi=300, bbox_inches='tight')
plt.show()

## 3.9 Segment-Channel ROI Matrix

In [ ]:
# Calculate ROI by segment and channel
if 'RFM_Segment' in df.columns:
    segment_roi = {}
    
    for segment in df['RFM_Segment'].unique():
        segment_data = df[df['RFM_Segment'] == segment]
        segment_roi[segment] = {}
        
        # Total revenue for segment
        segment_total_revenue = segment_data['MntTotal'].sum() if 'MntTotal' in segment_data.columns else 0
        
        # Total purchases for segment
        segment_web_purchases = segment_data['NumWebPurchases'].sum() if 'NumWebPurchases' in segment_data.columns else 0
        segment_catalog_purchases = segment_data['NumCatalogPurchases'].sum() if 'NumCatalogPurchases' in segment_data.columns else 0
        segment_store_purchases = segment_data['NumStorePurchases'].sum() if 'NumStorePurchases' in segment_data.columns else 0
        
        segment_total_purchases = segment_web_purchases + segment_catalog_purchases + segment_store_purchases
        
        # Distribute segment revenue proportionally
        if segment_total_purchases > 0:
            segment_web_revenue = segment_total_revenue * (segment_web_purchases / segment_total_purchases)
            segment_catalog_revenue = segment_total_revenue * (segment_catalog_purchases / segment_total_purchases)
            segment_store_revenue = segment_total_revenue * (segment_store_purchases / segment_total_purchases)
        else:
            segment_web_revenue = segment_catalog_revenue = segment_store_revenue = 0
        
        # Calculate ROI for each channel
        web_cost = segment_web_purchases * cost_per_transaction['Web']
        catalog_cost = segment_catalog_purchases * cost_per_transaction['Catalog']
        store_cost = segment_store_purchases * cost_per_transaction['Store']
        
        segment_roi[segment]['Web'] = ((segment_web_revenue - web_cost) / web_cost * 100) if web_cost > 0 else 0
        segment_roi[segment]['Catalog'] = ((segment_catalog_revenue - catalog_cost) / catalog_cost * 100) if catalog_cost > 0 else 0
        segment_roi[segment]['Store'] = ((segment_store_revenue - store_cost) / store_cost * 100) if store_cost > 0 else 0
    
    segment_roi_df = pd.DataFrame(segment_roi).T
    
    print("ROI (%) by Segment and Channel:")
    print(segment_roi_df.round(2))
    
    # Heatmap
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(segment_roi_df, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
                cbar_kws={'label': 'ROI (%)'}, ax=ax, linewidths=0.5)
    ax.set_title('ROI (%) by Customer Segment and Channel', fontsize=14, fontweight='bold', pad=20)
    ax.set_ylabel('Customer Segment', fontsize=12, fontweight='bold')
    ax.set_xlabel('Channel', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../visualizations/03_roi_by_segment_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Grouped bar chart
    fig, ax = plt.subplots(figsize=(12, 6))
    segment_roi_df.plot(kind='bar', ax=ax, width=0.8, alpha=0.8, edgecolor='black')
    ax.set_title('ROI by Customer Segment and Channel', fontsize=14, fontweight='bold', pad=20)
    ax.set_ylabel('ROI (%)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Customer Segment', fontsize=12, fontweight='bold')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
    ax.legend(title='Channel', fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('../visualizations/03_roi_by_segment_grouped.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Find best performing segment-channel combinations
    print("\\nTop 5 Best Performing Segment-Channel Combinations:")
    roi_stacked = segment_roi_df.stack().sort_values(ascending=False)
    for i, (combo, roi) in enumerate(roi_stacked.head(5).items(), 1):
        print(f"{i}. {combo[0]} + {combo[1]}: {roi:.2f}%")
else:
    print("RFM_Segment column not found")

## 3.10 Campaign Targeting Efficiency Analysis

In [ ]:
# Analyze which segment responds best to each campaign
if 'RFM_Segment' in df.columns:
    targeting_efficiency = {}
    
    for campaign in campaign_cols:
        if campaign in df.columns:
            segment_acceptance = df.groupby('RFM_Segment')[campaign].mean() * 100
            best_segment = segment_acceptance.idxmax()
            best_rate = segment_acceptance.max()
            overall_rate = (df[campaign].sum() / len(df)) * 100
            uplift = best_rate - overall_rate
            
            targeting_efficiency[campaign] = {
                'Best Segment': best_segment,
                'Best Segment Rate (%)': best_rate,
                'Overall Rate (%)': overall_rate,
                'Potential Uplift (%)': uplift
            }
    
    efficiency_df = pd.DataFrame(targeting_efficiency).T
    
    print("Campaign Targeting Efficiency Analysis:")
    print(efficiency_df.round(2))
    print("\\nInterpretation:")
    print("- 'Best Segment Rate' = Acceptance rate in most responsive segment")
    print("- 'Potential Uplift' = Performance gain if campaigns targeted only at best segments")
    
    # Visualization
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(efficiency_df))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, efficiency_df['Best Segment Rate (%)'], width, 
                   label='Best Segment Rate', color='green', alpha=0.7, edgecolor='black')
    bars2 = ax.bar(x + width/2, efficiency_df['Overall Rate (%)'], width, 
                   label='Overall Rate', color='steelblue', alpha=0.7, edgecolor='black')
    
    ax.set_ylabel('Acceptance Rate (%)', fontsize=12, fontweight='bold')
    ax.set_title('Campaign Targeting Efficiency: Best Segment vs Overall Performance', fontsize=14, fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(efficiency_df.index, rotation=45, ha='right')
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../visualizations/03_targeting_efficiency.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("RFM_Segment column not found")

## 3.11 Budget Reallocation Recommendations

In [ ]:
# Budget reallocation scenario
# Assume current budget is equally distributed
total_budget = 1000000  # $1M total budget
current_allocation = {
    'Web': total_budget * 0.33,
    'Catalog': total_budget * 0.33,
    'Store': total_budget * 0.34
}

print("\\nCURRENT BUDGET ALLOCATION (Assumed Equal Distribution):")
for channel, amount in current_allocation.items():
    print(f"{channel}: ${amount:,.0f} ({amount/total_budget*100:.1f}%)")

# Calculate recommended allocation based on ROI
roi_weights = roi_df['ROI (%)'].copy()
roi_weights = roi_weights[roi_weights > 0]  # Only positive ROI channels

recommended_allocation = {}
if len(roi_weights) > 0:
    roi_weights_normalized = roi_weights / roi_weights.sum()
    for channel in roi_weights.index:
        recommended_allocation[channel] = total_budget * roi_weights_normalized[channel]
    
    # Fill in zero-allocation channels if any
    for channel in ['Web', 'Catalog', 'Store']:
        if channel not in recommended_allocation:
            recommended_allocation[channel] = 0
else:
    recommended_allocation = current_allocation.copy()

print("\\nRECOMMENDED BUDGET ALLOCATION (Based on ROI):")
for channel in ['Web', 'Catalog', 'Store']:
    amount = recommended_allocation.get(channel, 0)
    print(f"{channel}: ${amount:,.0f} ({amount/total_budget*100:.1f}%)")

# Calculate expected impact
print(f"\\nEXPECTED IMPACT OF REALLOCATION:")

# Scale revenues based on budget changes
current_revenue = roi_df['Total Revenue ($)'].sum()
current_cost = roi_df['Total Cost ($)'].sum()

# Calculate scaling factors
scaling_factors = {}
for channel in ['Web', 'Catalog', 'Store']:
    current_cost_ch = roi_df.loc[channel, 'Total Cost ($)']
    if current_cost_ch > 0:
        scaling_factors[channel] = recommended_allocation[channel] / current_cost_ch
    else:
        scaling_factors[channel] = 0

# Calculate projected revenues
recommended_revenue = sum([roi_df.loc[ch, 'Total Revenue ($)'] * scaling_factors[ch] for ch in ['Web', 'Catalog', 'Store']])
recommended_cost = sum([roi_df.loc[ch, 'Total Cost ($)'] * scaling_factors[ch] for ch in ['Web', 'Catalog', 'Store']])

revenue_increase = recommended_revenue - current_revenue
revenue_increase_pct = (revenue_increase / current_revenue * 100) if current_revenue > 0 else 0

print(f"Current Expected Revenue: ${current_revenue:,.0f}")
print(f"Recommended Expected Revenue: ${recommended_revenue:,.0f}")
print(f"Projected Increase: ${revenue_increase:,.0f} ({revenue_increase_pct:+.1f}%)")

# Visualization: Budget allocation comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Current allocation
current_channels = list(current_allocation.keys())
current_values = list(current_allocation.values())
colors_allocation = ['#1f77b4', '#ff7f0e', '#2ca02c']

wedges1, texts1, autotexts1 = ax1.pie(current_values, labels=current_channels, autopct='%1.1f%%',
                                        colors=colors_allocation, startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'})
ax1.set_title('Current Budget Allocation', fontsize=13, fontweight='bold', pad=15)

# Recommended allocation
recommended_channels = list(recommended_allocation.keys())
recommended_values = list(recommended_allocation.values())
colors_recommended = [colors_allocation[current_channels.index(ch)] if ch in current_channels else '#666666' for ch in recommended_channels]

wedges2, texts2, autotexts2 = ax2.pie(recommended_values, labels=recommended_channels, autopct='%1.1f%%',
                                        colors=colors_recommended, startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'})
ax2.set_title(f'Recommended Budget Allocation\\n(Est. Revenue Increase: {revenue_increase_pct:.1f}%)', 
              fontsize=13, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('../visualizations/03_budget_reallocation.png', dpi=300, bbox_inches='tight')
plt.show()

# Waterfall-style comparison
fig, ax = plt.subplots(figsize=(12, 6))

channels_all = ['Web', 'Catalog', 'Store']
current_alloc_list = [current_allocation.get(ch, 0) for ch in channels_all]
recommended_alloc_list = [recommended_allocation.get(ch, 0) for ch in channels_all]

x = np.arange(len(channels_all))
width = 0.35

bars1 = ax.bar(x - width/2, current_alloc_list, width, label='Current Allocation', 
               color='steelblue', alpha=0.7, edgecolor='black')
bars2 = ax.bar(x + width/2, recommended_alloc_list, width, label='Recommended Allocation', 
               color='green', alpha=0.7, edgecolor='black')

ax.set_ylabel('Budget Allocation ($)', fontsize=12, fontweight='bold')
ax.set_title('Budget Reallocation: Current vs Recommended', fontsize=14, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(channels_all)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Format y-axis as currency
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'${height/1000:.0f}K',
                ha='center', va='bottom', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('../visualizations/03_budget_reallocation_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 3.12 Key Findings & Recommendations

In [ ]:
# Summary findings
print("\\n" + "="*80)
print("KEY FINDINGS & RECOMMENDATIONS")
print("="*80)

# Best campaign
best_campaign = acceptance_df.iloc[0]
print(f"\\n1. BEST PERFORMING CAMPAIGN:")
print(f"   Campaign: {best_campaign['Campaign']}")
print(f"   Acceptance Rate: {best_campaign['Acceptance Rate (%)']:.2f}%")

# Best channel by ROI
if len(roi_df) > 0:
    best_channel = roi_df['ROI (%)'].idxmax()
    best_roi = roi_df.loc[best_channel, 'ROI (%)']
    print(f"\\n2. BEST PERFORMING CHANNEL (ROI):")
    print(f"   Channel: {best_channel}")
    print(f"   ROI: {best_roi:.2f}%")
    print(f"   Revenue per $1 Spent: ${roi_df.loc[best_channel, 'Revenue per $1 Spent']:.2f}")

# Most responsive segment
if 'Total_Campaigns_Accepted' in df.columns and 'RFM_Segment' in df.columns:
    most_responsive = df.groupby('RFM_Segment')['Total_Campaigns_Accepted'].mean().idxmax()
    most_responsive_rate = df.groupby('RFM_Segment')['Total_Campaigns_Accepted'].mean().max()
    print(f"\\n3. MOST RESPONSIVE CUSTOMER SEGMENT:")
    print(f"   Segment: {most_responsive}")
    print(f"   Avg Campaigns Accepted: {most_responsive_rate:.2f}")

# Budget recommendations
print(f"\\n4. RECOMMENDED BUDGET CHANGES:")
for channel in ['Web', 'Catalog', 'Store']:
    current = current_allocation.get(channel, 0)
    recommended = recommended_allocation.get(channel, 0)
    change = recommended - current
    change_pct = (change / current * 100) if current > 0 else 0
    print(f"   {channel}: ${current:,.0f} → ${recommended:,.0f} ({change_pct:+.1f}%)")

# Expected impact
print(f"\\n5. EXPECTED IMPACT:")
print(f"   Projected Revenue Increase: ${revenue_increase:,.0f} ({revenue_increase_pct:+.1f}%)")

# Strategic recommendations
print(f"\\n6. STRATEGIC RECOMMENDATIONS:")
if len(roi_df) > 0:
    print(f"   a) Increase investment in high-ROI channels (e.g., {best_channel})")
if 'Total_Campaigns_Accepted' in df.columns and 'RFM_Segment' in df.columns:
    print(f"   b) Target {most_responsive} segment with priority campaigns")
print(f"   c) Focus campaign efforts on proven high-performers ({best_campaign['Campaign']})")
print(f"   d) Test and optimize low-performing channels before significant budget reductions")
print(f"   e) Implement segment-specific campaign strategies based on acceptance patterns")

print("\\n" + "="*80)

# Summary statistics
print("\\nSUMMARY STATISTICS:")
print(f"Total Customers Analyzed: {len(df):,}")
print(f"Total Campaigns Accepted: {df[campaign_cols].sum().sum():,.0f}")
print(f"Total Revenue: ${roi_df['Total Revenue ($)'].sum():,.0f}")
print(f"Total Marketing Cost: ${roi_df['Total Cost ($)'].sum():,.0f}")
print(f"Overall Net Profit: ${(roi_df['Total Revenue ($)'] - roi_df['Total Cost ($)']).sum():,.0f}")
if roi_df['Total Cost ($)'].sum() > 0:
    overall_roi = ((roi_df['Total Revenue ($)'].sum() - roi_df['Total Cost ($)'].sum()) / roi_df['Total Cost ($)'].sum() * 100)
    print(f"Overall ROI: {overall_roi:.2f}%")